INSTALL LIBRARIES

PLEASE USE CPU RUNTIME WITH MORE CPU CORES FOR THIS SECTION, TPU MAY BE USED, HAS MORE CORES THAN CPU OFFERED IN COLAB HERE

In [ ]:
### PLEASE CHANGE DIRECTORY AND ENSURE ALL RELEVANT INTERMEDIATE FILES ARE SAVED TO AN APPROPRIETE DIRECTORY, PREFERABLY YOUR GOOGLE DRIVE SO YOU DONT LOOSE ALL SAVED DATA, YOU WILL NEED TO CHANGE RUNTIME AT THE TRAINING SECTION

### CONTENT FOLDER OR DIRECTORY IS TEMPORAL AND SO YOU MAY LOOSE ALL DATA WHEN RUNTIME DISCONNECTS

In [ ]:
!pip install torch==2.4.0
!pip install torch_geometric
!pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/repo.html
!pip install numpy==1.24.4
!pip install rdkit
!pip install graphein[extras]

INPORT LIBRARIES AND FUNCTIONS

In [ ]:
import torch
import rdkit
import random
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from rdkit import Chem
from utils.graph_generator import run_graph_generator   # nake sure utils folder is present in the current woring directory

IMPORT AChE LABEL DATASET RETRIEVED FROM (Vignaux et al., 2023)

In [ ]:
m1 = pd.read_csv("/content/Human_dataset_1micromolar.csv")
ex = pd.read_csv("/content/Extended_human_dataset_100micromolar.csv")

In [ ]:
m1.head()

In [ ]:
ex.head()

In [ ]:
m = m1[["SMILES", "single-class-label"]]
m

In [ ]:
e = ex[["SMILES", "single-class-label"]]
e

In [ ]:
# count labels in each set

print(m["single-class-label"].value_counts())
print(e["single-class-label"].value_counts())

In [ ]:
# get actives in datasets

m_actives = m[m["single-class-label"] == 1]
e_actives = e[e["single-class-label"] == 1]


# inactives

m_inactives = m[m["single-class-label"] == 0]
e_inactives = e[e["single-class-label"] == 0]

In [ ]:
print(m_inactives.count())
print(e_inactives.count())
print(m_actives.count())
print(e_actives.count())

In [ ]:
t_act = pd.concat([m_actives, e_actives]).reset_index(drop=True)
t_inact = pd.concat([m_inactives, e_inactives]).reset_index(drop=True)

In [ ]:
t_act.head()

In [ ]:
t_inact.head()

In [ ]:
#check for duplicate smiles

print(t_act.duplicated(subset=["SMILES"]).sum())
print(t_inact.duplicated(subset=["SMILES"]).sum())

In [ ]:
# remove duplicates

t_actives = t_act.drop_duplicates(subset=["SMILES"])
t_inactives = t_inact.drop_duplicates(subset=["SMILES"])

In [ ]:
print(t_actives.count())
print(t_inactives.count())

In [ ]:
t_conc = pd.concat([t_actives, t_inactives]).reset_index(drop=True)
t_conc.head()

In [ ]:
# again check for duplicates

print(t_conc.duplicated(subset=["SMILES"]).sum())

In [ ]:
# cleanup duplicates but keep first

t_conc = t_conc.drop_duplicates(subset=["SMILES"], keep="first")
t_conc

In [ ]:
print(t_conc["single-class-label"].value_counts())

In [ ]:
# lets perform random selection to balance out the dataset

random.seed(42)

class_counts = t_conc["single-class-label"].value_counts()
minority_class_count = class_counts[1]

t_conc_minority = t_conc[t_conc["single-class-label"] == 1]
t_conc_majority = t_conc[t_conc["single-class-label"] == 0]

# Undersample the majority class

t_majority_undersampled = t_conc_majority.sample(n=minority_class_count, random_state=42)

# Combine the undersampled majority class with the minority class

t_balanced = pd.concat([t_majority_undersampled, t_conc_minority])

t = t_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
t

In [ ]:
print(t["single-class-label"].value_counts())

In [ ]:
# now lets split this randomly into train and external validation in 90:10

m, e = train_test_split(t, test_size=0.1, random_state=42)

In [ ]:
e

In [ ]:
m

In [ ]:
print(m["single-class-label"].value_counts())
print(e["single-class-label"].value_counts())

In [ ]:
# rename the labels column to y and SMILES to smiles

m_ = m.rename(columns={"single-class-label": "y", "SMILES": "smiles"})
e_ = e.rename(columns={"single-class-label": "y", "SMILES": "smiles"})

In [ ]:
m_.head()

In [ ]:
e_.head()

SAVE CLEAN FILES, YOU CAN JUST SAVE THE .csv AND LEAVE THE TENSOR OUT

In [ ]:
torch.save(m_, "/content/smiles_train.pt")
torch.save(e_, "/content/smiles_external.pt")

In [ ]:
m_.to_csv("/content/smiles_train.csv", index=False)
e_.to_csv("/content/smiles_external.csv", index=False)

TRAIN GRAPHS

In [ ]:
# NOTE: run_graph_generator returns tensor of PyTorch geometric Data object containing node features, edge indices, edge attributes and graph Laplacian positional encodings (PE) for each node in a graph.
# PE dimension capped at 51 atoms, thus k=51.

# smiles_list = Pandas Series of SMILES or list of SMILES with one SMILES per row
# output_dir = Output directory where the Data object will be written to. I recommend using a common directly for loadind and saving or writing files to. Some of the scripts have been configured to load from common directory.
# chunk_size = Number of CPU cores to use for parallel processing. Please specify this base on the resources available to you.
# run_graph_generator = Core function does performs the end-to-end computation

# OUTPUT FILE HAS NAME, Graph_Data.pt

In [ ]:
# PLEASE RENAME THE 'Graph_Data.pt' OUTPUT TO 'Graph_Data_train.pt'

if __name__ == "__main__":
    smiles_list = m_["smiles"]
    output_dir = "/content/"
    chunk_size = 5000
    num_processes = 20
    run_graph_generator(smiles_list, output_dir=output_dir, chunk_size=chunk_size, num_processes=num_processes)

In [ ]:
# LOAD AND INSPECT THE TRAINING DATA OBJECTS

d_t = torch.load("/content/Graph_Data_train.pt")
d_t

In [ ]:
print(len(d_t))

EXTERNAL

In [ ]:
# RENAME THE OUTPUT FILE 'Graph_Data.pt' TO 'Graph_Data_external.pt'

if __name__ == "__main__":
    smiles_list = e_["smiles"]
    output_dir = "/content/"
    chunk_size = 5000
    num_processes = 20
    run_graph_generator(smiles_list, output_dir=output_dir, chunk_size=chunk_size, num_processes=num_processes)

In [ ]:
# LOAD AND INSPEACT

d_e = torch.load("/content/Graph_Data_external.pt")
d_e

In [ ]:
print(len(d_e))

714


marge labels and graph data

In [ ]:
# TRAIN


labels = m["single-class-label"].reset_index(drop=True)

md = []
for i, data in tqdm(enumerate(d_t), total=len(d_t), desc="Adding labels to dataset"):
    data.y = torch.tensor([labels[i]], dtype=torch.float)

    md.append(data)

print(md[0])

In [ ]:
md

In [ ]:
len(md)

In [ ]:
# WE CAN NOW SAVE THE PREPARED AND READY TO BE TRAINED OR FED TO THE MODEL.

torch.save(md, "/content/Graph_Data_Train.pt")

In [ ]:
# EXTERNAL


labels = e["single-class-label"].reset_index(drop=True)

ed = []
for i, data in tqdm(enumerate(d_e), total=len(d_e), desc="Adding labels to dataset"):
    data.y = torch.tensor([labels[i]], dtype=torch.float)

    ed.append(data)

print(ed[0])

In [ ]:
ed

In [ ]:
len(ed)

In [ ]:
# WE CAN NOW SAVE THE PREPARED AND READY TO BE TRAINED OR FED TO THE MODEL

torch.save(ed, "/content/Graph_Data_External.pt")

ANALYZE DISTRIBUTION USING MACCS KEYS AS FEATURES FOR TRAINING AND EXTERNAL VALIDATION

In [ ]:
# train_df_path = Path to the SMILES of the training dataset.
# val_df_path = Path to the externa; validation dataset.
# tsne_plot_path = Path to where the t-SNE plot will automatically be saved to. PNG file is cool and recommended.
# analyze_dataset_diversity = The core function that performs all the MACCS keys computation, compute Tanimoto similarities and then perform the t-SNE analysis with the plot as well

# Please ensure your training dataset is a pandas dataframe with SMILES column header as "smiles" and the labels header as "y". Anything different from this wont work, or you need to modify the headers identifiers from the analyze-distribution.py from the nutils folder

In [ ]:
if __name__ == "__main__":
    analyze_dataset_diversity(
        train_df_path="/content/smiles_train.csv",
        val_df_path="/content/smiles_extended.csv",
        tsne_plot_path="/content/tsne_maccs_plot.png"
    )

In [ ]:
# lets load the png image and analyze

Image(filename='/content/tsne_maccs_plot.png')

TRAIN AChE MODEL USING TRANSFER LEARNING

PLEASE CHANGE RUNTIME TO GPU FOR FASTER TRAINING

INSTALL PACKAGES FOR THIS SECTION

In [ ]:
!pip install torch==2.4.0
!pip install torch_geometric
!pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/repo.html
!pip install numpy==1.24.4
!pip install rdkit
!pip install graphein[extras]

IMPORT LIBRARIES AND FUNCTIONS

In [ ]:

import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image
from torch_geometric.data import Data
from utils.run_inference_for_classification import run_inference
from utils.analyze_distribution import analyze_dataset_diversity
from utils.fine_tuning_chemgraphormer_script_classifier import run_finetuning

SPLIT TRAINING GRAPH DATA, md INTO TRAIN:VALIATION IN RATIO OF 85:15

In [ ]:
# FIRST LOAD THE TRAINING GRAPH DATASET

md = torch.load("/content/Graph_Data_Train.pt")
md

In [ ]:
print(len(md))

In [ ]:
# NOW WE CAN SPLIT

random.seed(42)
indices = list(range(len(md)))
random.shuffle(indices)

train_end = int(0.85 * len(md))
train_indices = indices[:train_end]
valid_indices = indices[train_end:]

m_train = [md[i] for i in train_indices]
m_valid = [md[i] for i in valid_indices]

In [ ]:
print(len(m_train))
print(len(m_valid))

In [ ]:
m_train

In [ ]:
m_valid

In [ ]:
# SAVE THE SPLIT FILES

torch.save(m_train, "/content/Graph_Data_Train.pt")   # THIS IS NOW THE TRAINING DATASET
torch.save(m_valid, "/content/Graph_Data_Valid.pt")   # THIS IS THE DATASET FOR INTERVAL VALIDATION

RUN TRAINING

In [ ]:
# train_path = Path to the final prepared training dataset (the split one-85%).
# valid_path = Path to the final internal validation dataset (the split one-15%).
# encoder_ckpt_path = Path to the pre-trained ChemGraphormer encoder checkpoint. This is the fused encoder model and can be found in the outputs folder
# finetune_ckpt_dir = Directory to save the fine-tuned model checkpoints, thus what we are going to train now, AChEi-PRED.
# finetune_log_csv_path = Path to save the training logs in CSV format.
# batch_size = Number of samples per batch during training. Please use small batch sizes cos this is a fine-tuning model with very few dataset and we need very small steps to make proper and effective gradient updates of model weights
# epochs = Total number of training epochs.
# device = Device to use for training. Recommend cuda for faster execution here.
# num_classes = Number of output classes for classification (2 for binary classification). The script is designed to handle multiclass classifications but then you need a very minimal ajustment in the fine_tuning_chemgraphormer_script_classifier script in the utils folder.
# n_heads = Number of attention heads in the model. Leave this at default if you can, in order to match the pre-train architecture.
# num_layers = Number of layers in the model. Also leave this at default if you can.
# lr = Learning rate for the optimizer. Please use a small learning rate for this fine-tuning phase, but too small may also prevent effective learning, keep this at default or or you may experiment with different paramter values to see which works best for your task.
# dropout = Dropout rate for regularization. Even though all the encoder top layers have been frozen from the fine_tuning_chemgraphormer_script_classifier script, leaving only the classification head to adapt to this new dataset. The number of paramters in the head is quite huge for classification with this number of samples and so we need a little larger dropout, which we set to 0.3 to prevent orverfitting in the head.
# patience = Number of epochs to wait for improvement before early stopping.
# min_delta = Minimum change in the monitored metric to qualify as an improvement.

# run_finetuning = The core function with fine_tuning_chemgraphormer_script_classifier that performs all the training process

# COS THE MCCS METRIC IS A UNIFIED ONE, INTEGRATING ALL THE 4 METRICS IN THE CONFUSION MATRIX, I SET THAT AS THE METRIC TO MONITOR TRAINING AND FOR MODEL SELECTION.

In [ ]:
# Binary classification with cross entropy loss for optimization

if __name__ == "__main__":
    run_finetuning(
        train_path="/content/Graph_Data_Train.pt",
        valid_path="/content/Graph_Data_Valid.pt",
        encoder_ckpt_path="/content/fused_chemgraphormer_encoder.pt",
        finetune_ckpt_dir="/content/",
        finetune_log_csv_path="/content/finetune_logs.csv",
        batch_size=8,
        epochs=150,
        device="cuda",
        num_classes=2,
        n_heads=16,
        num_layers=8,
        lr=1e-4,
        dropout=0.3,
        patience=15,
        min_delta=8e-5
    )

EXTERNAL VALIDATION

In [ ]:
# model_path = Path to where the fine-tuned model was saved. Should be found at the finetune_ckpt_path.
# test_path = Path to where the external dataset was saved at.
# output_csv_path = Path to where the training logs will be written to
# device to use for the inference. cuda is mostly recommended if available cos its super fast
# batch_size = Number of batches to process at a time. This time, the model is in the evaluaion mode so you can set this high if you want, it wont affect results in anyway
# num_classes = Number of classes predicting. Should match what was used for the training and so set it to two as in the default if binary classification. You have to change this if adapted for multiclass problem.

In [ ]:
if __name__ == "__main__":
    run_inference(
        model_path="/content/chemgraphormer_finetuned_best.pt",
        test_path="/content/Graph_Data_External.pt",
        output_csv_path="/content/test_predictions_ache.csv",
        device="cuda",
        batch_size=8,
        num_classes=2
    )

ANALYSIS

In [ ]:
# LOAD THE FINE-TUNED MODEL TRAINING LOGS

fin = pd.read_csv("/content/finetune_logs.csv")
fin

In [ ]:
plt.figure(figsize=(10,6))

# Subplot A for loss
plt.subplot(2, 2, 1) # 2 rows, 2 columns, first plot
plt.plot(fin['Train Loss'], label='train_loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.title('A')
plt.legend()
plt.grid(visible=False)

# Subplot B for Accuracy, Precision, Recall and F1
plt.subplot(2, 2, 2) # 2 rows, 2 columns, second plot
plt.plot(fin['Val Accuracy'], label='Accuracy')
plt.plot(fin['Val Precision'], label='Precision')
plt.plot(fin['Val Recall'], label='Recall')
plt.plot(fin['Val F1'], label='F1')
plt.ylabel('Metric Value')
plt.xlabel('Epochs')
plt.title('B')
plt.legend()
plt.grid(visible=False)

# Subplot C for Specificity and Kappa
plt.subplot(2, 2, 3) # 2 rows, 2 columns, third plot
plt.plot(fin['Val Specificity'], label='Specificity')
plt.plot(fin['Val Kappa'], label='Kappa')
plt.ylabel('Metric Value')
plt.xlabel('Epochs')
plt.title('C')
plt.legend()
plt.grid(visible=False)


# Subplot D for AUC and MCC
plt.subplot(2, 2, 4) # 2 rows, 2 columns, fourth plot
plt.plot(fin['Val AUC'], label='AUC')
plt.plot(fin['Val MCC'], label='MCC')
plt.ylabel('Metric Value')
plt.xlabel('Epochs')
plt.title('D')
plt.legend()
plt.grid(visible=False)


plt.tight_layout()
plt.show()